# 전처리 (Preprocessing) — 전체 데이터셋 통합

서울DF 프로젝트에서 사용하는 **모든 데이터셋**에 대한 전처리를 정리합니다.
각 항목마다 **어떤 방법을 썼는지**와 **왜 그 기준을 적용했는지**를 함께 작성합니다.

## 대상 데이터셋 (8종)
| 번호 | 데이터 | 주요 분석 파일 |
|------|--------|----------------|
| 1 | 추정매출-상권 | dada, deep, 페르소나 A·B·C·D |
| 2 | 영역-상권 | 모든 파일 (상권코드 매핑) |
| 3 | 길단위인구-상권 | jeongyeon_eda, dada, deep |
| 4 | 직장인구-상권 | jiwoo_직장점심, nagyung, deep |
| 5 | 상주인구-상권 | jiwoo_50대이상, nagyung, deep |
| 6 | 상권변화지표-상권 | nagyung_eda, jeongyeon_eda, dada |
| 7 | 점포-상권 | sohee_eda, nagyung_취약상권 |
| 8 | 지하철 승하차 / 소득 | jeongyeon_deep_data2, jeongyeon_deep_data3 |

## 전처리 항목 요약
| 항목 | 핵심 방법 |
|------|-----------|
| 결측치 | `isnull().sum()` → `fillna(0)` / `fillna('서울 외')` / `dropna(subset=[...])` |
| 이상치 | `describe()` / IQR 방법 / 분기 필터링 / 최소 등장 기준 필터링 |
| 중복 | `duplicated().sum()` → `drop_duplicates(subset=복합키)` |
| 데이터 타입 변환 | `astype()` / `pd.to_numeric(errors='coerce')` / `pd.PeriodIndex` / 특수문자 정리 |
| 데이터 변환 | 연도·분기 추출 / 비율 피처 / groupby 집계 / merge / 구간화 / 표준화 / 좌표 변환 |


## 0. 공통 데이터 로딩

In [1]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path(".") if Path("./data").exists() else Path("..")

# ── 1. 추정매출 (전체 연도 합산)
DATA_DIR = str(project_root / "data/extracted/서울시_상권분석서비스(추정매출+영역)/") + "/"
df_sales = pd.concat(
    [pd.read_csv(f, encoding="cp949", low_memory=False)
     for f in sorted(glob.glob(DATA_DIR + "*추정매출*.csv"))],
    ignore_index=True
)

# ── 2. 영역
df_area = pd.read_csv(
    project_root / "data/extracted/서울시_상권분석서비스(추정매출+영역)/서울시 상권분석서비스(영역-상권).csv",
    encoding="euc-kr"
)

# ── 3. 길단위인구
df_pop = pd.read_csv(
    project_root / "data/extracted/서울시 상권분석서비스(길단위인구-상권)/서울시 상권분석서비스(길단위인구-상권).csv",
    encoding="cp949"
)

# ── 4. 직장인구
df_work = pd.read_csv(
    project_root / "data/06_직장인구/서울시 상권분석서비스(직장인구-상권).csv",
    encoding="cp949"
)

# ── 5. 상주인구
df_resident = pd.read_csv(
    project_root / "data/07_상주인구/서울시 상권분석서비스(상주인구-상권).csv",
    encoding="cp949"
)

# ── 6. 상권변화지표
df_change = pd.read_csv(
    project_root / "data/08_상권변화지표/서울시 상권분석서비스(상권변화지표-상권).csv",
    encoding="cp949"
)

# ── 7. 점포 (연도별 파일 합산)
_store_files = sorted(glob.glob(str(project_root / "data/extracted/상권분석서비스(점포_상권)/*.csv")))
df_store = pd.concat(
    [pd.read_csv(f, encoding="cp949", low_memory=False) for f in _store_files],
    ignore_index=True
) if _store_files else pd.DataFrame()

# ── 8. 소득소비
df_income = pd.read_csv(
    project_root / "data/extracted/서울시_외부데이터추가/서울시 상권분석서비스(소득소비-상권).csv",
    encoding="cp949"
)

# ── 9. 지하철 승하차
df_subway = pd.read_csv(
    project_root / "data/extracted/서울시_외부데이터추가/서울시 지하철 호선별 역별 시간대별 승하차 인원 정보 (1).csv",
    encoding="cp949"
)

print("✅ 데이터 로드 완료")
for name, df in [("추정매출", df_sales), ("영역", df_area), ("길단위인구", df_pop),
                  ("직장인구", df_work), ("상주인구", df_resident), ("상권변화지표", df_change),
                  ("점포", df_store), ("소득소비", df_income), ("지하철", df_subway)]:
    print(f"  {name}: {df.shape}")


✅ 데이터 로드 완료
  추정매출: (519931, 55)
  영역: (1650, 11)
  길단위인구: (46184, 27)
  직장인구: (45840, 26)
  상주인구: (40812, 29)
  상권변화지표: (46200, 11)
  점포: (1831925, 14)
  소득소비: (45590, 17)
  지하철: (81111, 52)


## 1. 결측치 (Missing Values)

In [2]:
# ══════════════════════════════════════════════════════════
# [전처리 1] 결측치 확인 및 처리
# ══════════════════════════════════════════════════════════

datasets = {
    '추정매출':     df_sales,
    '영역':        df_area,
    '길단위인구':  df_pop,
    '직장인구':    df_work,
    '상주인구':    df_resident,
    '상권변화지표': df_change,
    '점포':        df_store,
}

print("=== 데이터셋별 결측치 현황 ===")
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_cols = missing[missing > 0]
    if len(missing_cols) == 0:
        print(f"  [{name}] 결측치 없음 ({df.shape[0]:,}행)")
    else:
        print(f"  [{name}] 결측 컬럼 {len(missing_cols)}개:")
        for col, cnt in missing_cols.items():
            print(f"    - {col}: {cnt:,}개 ({cnt/len(df)*100:.1f}%)")

# ──────────────────────────────────────────────────────────
# 처리 방법별 기준
# ──────────────────────────────────────────────────────────

# [방법 1] fillna(0) — 업종 매출 컬럼
# 기준: 상권에 해당 업종이 없으면 매출 자체가 0 → 결측이 아니라 0원을 의미
매출_금액_cols = [c for c in df_sales.columns if '매출_금액' in c or '매출_건수' in c]
df_sales[매출_금액_cols] = df_sales[매출_금액_cols].fillna(0)
print(f"\n[방법 1] 매출 컬럼 fillna(0) 적용: {len(매출_금액_cols)}개 컬럼")

# [방법 2] fillna('서울 외') — 지하철역 자치구 매핑 누락
# 기준: 지하철 승하차 데이터에서 역명→자치구 매핑 사전에 없는 역(서울 외 역)
# 코드 예시 (jeongyeon_eda.ipynb):
#   df['자치구'] = df['자치구'].fillna('서울 외')
print("[방법 2] 지하철역 자치구 미매핑 → fillna('서울 외') (jeongyeon_eda 적용)")

# [방법 3] dropna(subset=[...]) — 소득 데이터 결측 행 제거
# 기준: 월_평균_소득_금액이 없는 상권은 소득 분석 자체가 불가 → 해당 행 제거
# 코드 예시 (jeongyeon_deep_data3.ipynb):
#   df_final3 = df_final3.dropna(subset=['월_평균_소득_금액'])
print("[방법 3] 소득 결측 행 dropna(subset=['월_평균_소득_금액']) (jeongyeon_deep_data3 적용)")

# [방법 4] replace(0, np.nan) — 비율 계산 시 분모=0 방지
# 기준: 총매출·총유동인구가 0인 경우 비율 정의 불가 → inf 발생 방지
safe = df_sales['당월_매출_금액'].replace(0, np.nan)
print("[방법 4] 총매출=0 상권 → replace(0, np.nan) 후 비율 계산 (0 나누기 방지)")

# [방법 5] left join 후 결측 확인 — merge 누락 탐지
# 기준: 상권변화지표와 직장인구 데이터를 merge할 때 누락 상권 발생 가능
# 코드 예시 (deep.ipynb):
#   missing_rows = df_merged[df_merged['총_직장_인구_수'].isna()]
print("[방법 5] merge left join 후 NaN 상권 확인 (deep.ipynb 적용)")

# [방법 6] notna().all(axis=1) — RFM 지표 NaN 행 필터링
# 기준: R·F·M 3개 지표 중 하나라도 없으면 RFM 점수 산출 불가 → 제외
# 코드 예시 (RFM_EDA_진행.ipynb):
#   rfm_clean = rfm[rfm[['R_매출증감률', 'F_결제건수', 'M_객단가']].notna().all(axis=1)]
print("[방법 6] RFM 지표 3개 동시 결측 행 제거 (RFM_EDA_진행 적용)")

print("\n✅ 결측치 처리 완료")


=== 데이터셋별 결측치 현황 ===
  [추정매출] 결측치 없음 (519,931행)
  [영역] 결측치 없음 (1,650행)
  [길단위인구] 결측치 없음 (46,184행)
  [직장인구] 결측치 없음 (45,840행)
  [상주인구] 결측치 없음 (40,812행)
  [상권변화지표] 결측치 없음 (46,200행)
  [점포] 결측치 없음 (1,831,925행)

[방법 1] 매출 컬럼 fillna(0) 적용: 48개 컬럼
[방법 2] 지하철역 자치구 미매핑 → fillna('서울 외') (jeongyeon_eda 적용)
[방법 3] 소득 결측 행 dropna(subset=['월_평균_소득_금액']) (jeongyeon_deep_data3 적용)
[방법 4] 총매출=0 상권 → replace(0, np.nan) 후 비율 계산 (0 나누기 방지)
[방법 5] merge left join 후 NaN 상권 확인 (deep.ipynb 적용)
[방법 6] RFM 지표 3개 동시 결측 행 제거 (RFM_EDA_진행 적용)

✅ 결측치 처리 완료


## 2. 이상치 (Outliers)

In [3]:
# ══════════════════════════════════════════════════════════
# [전처리 2] 이상치 확인 및 처리
# ══════════════════════════════════════════════════════════

# ── [방법 1] describe() 기술통계 확인 ─────────────────────
# 기준: 모든 데이터셋에 대해 min·max·mean·std를 확인하여 비상식적 값 탐지
print("=== 추정매출 핵심 수치 기술통계 ===")
check_cols = ['당월_매출_금액', '당월_매출_건수']
존재_cols = [c for c in check_cols if c in df_sales.columns]
print(df_sales[존재_cols].describe().map(lambda x: f'{x:,.0f}').to_string())

print("\n=== 상권변화지표 기술통계 ===")
ops_cols = ['운영_영업_개월_평균', '폐업_영업_개월_평균']
존재_ops = [c for c in ops_cols if c in df_change.columns]
if 존재_ops:
    print(df_change[존재_ops].describe().to_string())
    # 0개월 → 데이터 오류 가능성 (실제 영업 중인데 0개월은 비상식적)
    zero_ops = (df_change[존재_ops] == 0).sum()
    print(f"  0개월 값 개수: {zero_ops.to_dict()}")

# ── [방법 2] IQR 방법 — RFM 지표 극단치 탐지 ──────────────
# 기준: Q1-1.5*IQR ~ Q3+1.5*IQR 범위 밖을 극단치로 판단
# 사용: RFM_EDA_진행.ipynb — R(매출증감률)·F(결제건수)·M(객단가) 이상치 탐지
def find_outliers_iqr(series, name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"  [{name}] IQR범위: {lower:,.0f} ~ {upper:,.0f} | 극단치: {len(outliers)}개 ({len(outliers)/len(series)*100:.1f}%)")
    return outliers

print("\n=== IQR 방법 극단치 탐지 (추정매출 기준) ===")
per_상권 = df_sales.groupby('상권_코드')['당월_매출_금액'].sum()
find_outliers_iqr(per_상권, '상권별 총매출')

# ── [방법 3] 이상치 제거 안 함 — 상권 매출 ─────────────────
# 기준: 강남 오피스 상권 vs 골목 상권은 매출 규모가 100배 이상 차이 → 정상 데이터
#       극단값 자체가 분석의 핵심 대상(고매출 상권 vs 저매출 상권 비교)
#       → 제거 대신 로그 변환으로 분포 치우침 보정
print("\n[방법 3] 매출 이상치 제거 안 함 → 로그 변환으로 분포 보정")
per_상권_log = np.log(per_상권.replace(0, np.nan))
print(f"  원본 범위: {per_상권.min():,.0f} ~ {per_상권.max():,.0f}")
print(f"  로그 범위: {per_상권_log.min():.1f} ~ {per_상권_log.max():.1f}")

# ── [방법 4] 분기 필터링 — 2025 미완성 데이터 제외 ──────────
# 기준: 2025년 데이터는 분석 시점에 일부 분기만 존재 → 연도 간 비교 왜곡 방지
#       적용: nagyung_eda, nagyung_관광특구, jeongyeon_eda 등
df_change_filtered = df_change[df_change['기준_년분기_코드'] < 20250].copy()
print(f"\n[방법 4] 2025 분기 제외: {len(df_change):,}행 → {len(df_change_filtered):,}행")

# ── [방법 5] 최소 등장 분기 필터링 — 취약상권 분석 ────────────
# 기준: 8분기(2년) 미만으로 등장하는 상권은 데이터 부족 → 신뢰도 낮아 제외
#       적용: nagyung_eda, dada.ipynb (취약상권 후보 추출)
if '상권_코드_명' in df_change_filtered.columns:
    분기수_per_상권 = df_change_filtered.groupby('상권_코드_명').size()
    충분한_상권 = 분기수_per_상권[분기수_per_상권 >= 8].index
    print(f"\n[방법 5] 8분기 이상 등장 상권: {len(충분한_상권):,}개 / 전체 {분기수_per_상권.nunique():,}개")

# ── [방법 6] 취약상권 80~99% 기준 필터링 ──────────────────────
# 기준: 전체 분기 중 '상권축소' 비율이 80% 이상인 상권을 취약상권으로 정의
#       100%는 데이터가 극히 적거나 폐업 완료 상권일 가능성 → 분석 제외
if '상권_변화_지표_명' in df_change_filtered.columns and '상권_코드_명' in df_change_filtered.columns:
    total = df_change_filtered.groupby('상권_코드_명').size()
    축소 = df_change_filtered[df_change_filtered['상권_변화_지표_명'] == '상권축소'].groupby('상권_코드_명').size()
    축소_비율 = (축소 / total * 100).dropna()
    취약 = 축소_비율[(축소_비율 >= 80) & (축소_비율 < 100)]
    print(f"\n[방법 6] 취약상권 후보(80~99%): {len(취약)}개")

print("\n✅ 이상치 처리 완료")


=== 추정매출 핵심 수치 기술통계 ===
                당월_매출_금액    당월_매출_건수
count            519,931     519,931
mean       1,001,650,302      34,300
std        9,659,729,281     160,595
min                   31           1
25%           44,074,782         799
50%          166,349,258       4,237
75%          596,072,470      21,933
max    1,373,912,008,223  17,041,961

=== 상권변화지표 기술통계 ===
        운영_영업_개월_평균   폐업_영업_개월_평균
count  46200.000000  46200.000000
mean     105.046061     51.436169
std       25.399291     10.175792
min        0.000000     17.000000
25%       90.000000     46.000000
50%      101.000000     50.000000
75%      116.000000     55.000000
max      351.000000    226.000000
  0개월 값 개수: {'운영_영업_개월_평균': 4, '폐업_영업_개월_평균': 0}

=== IQR 방법 극단치 탐지 (추정매출 기준) ===
  [상권별 총매출] IQR범위: -283,033,468,583 ~ 529,037,538,471 | 극단치: 208개 (13.0%)

[방법 3] 매출 이상치 제거 안 함 → 로그 변환으로 분포 보정
  원본 범위: 4,122,092 ~ 24,152,804,837,064
  로그 범위: 15.2 ~ 30.8

[방법 4] 2025 분기 제외: 46,200행 → 39,600행

[방법 5] 8분기 이상 등장 상권: 1

## 3. 중복 (Duplicates)

In [4]:
# ══════════════════════════════════════════════════════════
# [전처리 3] 중복 확인 및 처리
# ══════════════════════════════════════════════════════════

datasets_dup = {
    '추정매출':     df_sales,
    '영역':        df_area,
    '길단위인구':  df_pop,
    '직장인구':    df_work,
    '상주인구':    df_resident,
    '상권변화지표': df_change,
}
if not df_store.empty:
    datasets_dup['점포'] = df_store

print("=== 데이터셋별 중복 행 현황 ===")
for name, df in datasets_dup.items():
    n_dup = df.duplicated().sum()
    flag = "⚠️ 중복 있음" if n_dup > 0 else "✓"
    print(f"  [{name}] 중복 행: {n_dup:,}개  {flag}")

# ── 처리 기준 ──────────────────────────────────────────────

# [추정매출] 복합키: (기준_년분기_코드, 상권_코드, 서비스_업종_코드_명)
# 기준: 동일 분기·상권·업종이 두 번 나오면 groupby 집계 시 매출 이중 합산 오류
key_매출 = ['기준_년분기_코드', '상권_코드', '서비스_업종_코드_명']
존재_key = [c for c in key_매출 if c in df_sales.columns]
if 존재_key:
    n_before = len(df_sales)
    df_sales_dedup = df_sales.drop_duplicates(subset=존재_key)
    print(f"\n[추정매출] 복합키 중복 제거: {n_before:,}행 → {len(df_sales_dedup):,}행 (제거: {n_before - len(df_sales_dedup):,}행)")

# [상권변화지표] 복합키: (기준_년분기_코드, 상권_코드)
# 기준: 동일 분기·상권에 변화지표가 두 번 나오면 안 됨
key_change = ['기준_년분기_코드', '상권_코드']
if all(c in df_change.columns for c in key_change):
    n_before = len(df_change)
    df_change_dedup = df_change.drop_duplicates(subset=key_change)
    print(f"[상권변화지표] 복합키 중복 제거: {n_before:,}행 → {len(df_change_dedup):,}행 (제거: {n_before - len(df_change_dedup):,}행)")

# [지하철 데이터] drop_duplicates() — jeongyeon_deep_data2
# 기준: 지하철 승하차 데이터에서 동일 역·날짜 중복 가능
print("[지하철] duplicated().sum() 확인 후 drop_duplicates() 적용 (jeongyeon_deep_data2)")

print("\n✅ 중복 처리 완료")


=== 데이터셋별 중복 행 현황 ===
  [추정매출] 중복 행: 0개  ✓
  [영역] 중복 행: 0개  ✓
  [길단위인구] 중복 행: 0개  ✓
  [직장인구] 중복 행: 0개  ✓
  [상주인구] 중복 행: 0개  ✓
  [상권변화지표] 중복 행: 0개  ✓
  [점포] 중복 행: 0개  ✓

[추정매출] 복합키 중복 제거: 519,931행 → 519,931행 (제거: 0행)
[상권변화지표] 복합키 중복 제거: 46,200행 → 46,200행 (제거: 0행)
[지하철] duplicated().sum() 확인 후 drop_duplicates() 적용 (jeongyeon_deep_data2)

✅ 중복 처리 완료


## 4. 데이터 타입 변환 (Data Type Conversion)

In [5]:
# ══════════════════════════════════════════════════════════
# [전처리 4] 데이터 타입 변환
# ══════════════════════════════════════════════════════════

print("=== 주요 컬럼 dtype 확인 ===")
key_cols_check = ['기준_년분기_코드', '상권_코드']
for name, df in [('추정매출', df_sales), ('길단위인구', df_pop), ('직장인구', df_work)]:
    exist = [c for c in key_cols_check if c in df.columns]
    if exist:
        dtypes_str = ', '.join([f"{c}:{df[c].dtype}" for c in exist])
        print(f"  [{name}] {dtypes_str}")

# ── [방법 A] 기준_년분기_코드 → 연도·분기 분리 ──────────────
# 기준: 원본이 YYYYQ(5자리 정수, 예:20254)로 제공 → 시계열 분석용 연도·분기 필요
# 방법1: astype(str) + str 슬라이싱 (가독성 높음)
# 방법2: //10, %10 정수 나눗셈 (성능 빠름)

print("\n=== [A] 기준_년분기_코드 파싱 ===")
for df_name, df in [('추정매출', df_sales), ('상권변화지표', df_change), ('길단위인구', df_pop)]:
    if '기준_년분기_코드' in df.columns:
        df['연도'] = df['기준_년분기_코드'] // 10           # 방법2: 20254→2025
        df['분기'] = df['기준_년분기_코드'] % 10            # 방법2: 20254→4
        df['년분기'] = df['연도'].astype(str) + '-Q' + df['분기'].astype(str)  # "2025-Q4"
        print(f"  [{df_name}] 연도 범위: {df['연도'].min()}~{df['연도'].max()}")

# ── [방법 B] 사용월 (YYYYMM) → 연도·월·분기 ─────────────────
# 기준: 지하철 승하차·카드 이용 데이터의 날짜 형식이 YYYYMM(6자리)
#       월에서 분기 계산: 분기 = (월-1)//3 + 1
# 적용: jeongyeon_eda.ipynb, jeongyeon_deep_data2.ipynb
print("\n=== [B] 사용월(YYYYMM) → 연도·분기 변환 예시 ===")
sample_사용월 = pd.Series([202401, 202402, 202403, 202404, 202412], dtype=int)
사용월_str = sample_사용월.astype(str)
연도 = 사용월_str.str[:4].astype(int)
월   = 사용월_str.str[4:6].astype(int)
분기 = (월 - 1) // 3 + 1
연도분기 = (연도.astype(str) + 분기.astype(str)).astype(int)
print(pd.DataFrame({'사용월': sample_사용월, '연도': 연도, '월': 월, '분기': 분기, '연도분기': 연도분기}))

# ── [방법 C] 비숫자 문자열 → 숫자 강제 변환 ────────────────
# 기준: 방문수 등 외부 연계 데이터에 '-', '없음' 혼재 → pd.to_numeric(errors='coerce')
# 적용: jeongyeon_eda.ipynb (방문수 컬럼)
print("\n=== [C] 비숫자 컬럼 → 숫자 강제 변환 ===")
sample = pd.Series(['1000', '2500', '-', '3100', '없음', '4200'])
converted = pd.to_numeric(sample, errors='coerce')
print(f"  변환 전: {sample.tolist()}")
print(f"  변환 후: {converted.tolist()}")
print(f"  NaN 개수: {converted.isna().sum()}")

# ── [방법 D] 시계열 Period 타입 변환 ────────────────────────
# 기준: pd.PeriodIndex(freq='Q')로 변환하면 시계열 shift(), diff() 연산 가능
# 적용: jeongyeon_eda.ipynb (분기별 시계열 연속성 분석)
print("\n=== [D] pd.PeriodIndex 변환 ===")
if '연도' in df_change.columns and '분기' in df_change.columns:
    period_idx = pd.PeriodIndex(
        df_change['연도'].astype(str) + 'Q' + df_change['분기'].astype(str), freq='Q'
    )
    print(f"  Period 샘플: {period_idx.unique()[:4].tolist()}")

# ── [방법 E] 특수문자 정리 — 상권코드명 ─────────────────────
# 기준: 일부 상권 이름에 '?' 등 인코딩 깨진 특수문자 → 시각화 레이블 오류
# 적용: jeongyeon_eda.ipynb
# df['상권_코드_명'] = df['상권_코드_명'].str.replace('?', '·', regex=False)
print("\n=== [E] 상권명 특수문자 정리 ===")
sample_names = pd.Series(['을지로?입구', '종로?3가', '홍대입구'])
cleaned = sample_names.str.replace('?', '·', regex=False)
print(f"  정리 전: {sample_names.tolist()}")
print(f"  정리 후: {cleaned.tolist()}")

print("\n✅ 데이터 타입 변환 완료")


=== 주요 컬럼 dtype 확인 ===
  [추정매출] 기준_년분기_코드:int64, 상권_코드:int64
  [길단위인구] 기준_년분기_코드:int64, 상권_코드:int64
  [직장인구] 기준_년분기_코드:int64, 상권_코드:int64

=== [A] 기준_년분기_코드 파싱 ===
  [추정매출] 연도 범위: 2020~2025
  [상권변화지표] 연도 범위: 2019~2025
  [길단위인구] 연도 범위: 2019~2025

=== [B] 사용월(YYYYMM) → 연도·분기 변환 예시 ===
      사용월    연도   월  분기   연도분기
0  202401  2024   1   1  20241
1  202402  2024   2   1  20241
2  202403  2024   3   1  20241
3  202404  2024   4   2  20242
4  202412  2024  12   4  20244

=== [C] 비숫자 컬럼 → 숫자 강제 변환 ===
  변환 전: ['1000', '2500', '-', '3100', '없음', '4200']
  변환 후: [1000.0, 2500.0, nan, 3100.0, nan, 4200.0]
  NaN 개수: 2

=== [D] pd.PeriodIndex 변환 ===
  Period 샘플: [Period('2025Q4', 'Q-DEC'), Period('2025Q3', 'Q-DEC'), Period('2025Q2', 'Q-DEC'), Period('2025Q1', 'Q-DEC')]

=== [E] 상권명 특수문자 정리 ===
  정리 전: ['을지로?입구', '종로?3가', '홍대입구']
  정리 후: ['을지로·입구', '종로·3가', '홍대입구']

✅ 데이터 타입 변환 완료


## 5. 데이터 변환 (Data Transformation)

In [6]:
# ══════════════════════════════════════════════════════════
# [전처리 5] 데이터 변환
# ══════════════════════════════════════════════════════════

from sklearn.preprocessing import StandardScaler

# ── (A) 비율(%) 피처 생성 ─────────────────────────────────
# 기준: 상권 규모 차이가 크므로 절대 매출 대신 비중으로 비교
# 적용: 모든 페르소나(A·B·C·D)·dada·deep 파일

g = df_sales.groupby('상권_코드').agg(
    총매출   = ('당월_매출_금액', 'sum'),
    총건수   = ('당월_매출_건수', 'sum'),
    점심매출 = ('시간대_11~14_매출_금액', 'sum'),
    저녁매출 = ('시간대_17~21_매출_금액', 'sum'),
    야간매출 = ('시간대_21~24_매출_금액', 'sum'),
    주말매출 = ('주말_매출_금액', 'sum'),
    여성매출 = ('여성_매출_금액', 'sum'),
    남성매출 = ('남성_매출_금액', 'sum'),
).reset_index()

safe_tot = g['총매출'].replace(0, np.nan)
g['점심비중']  = g['점심매출'] / safe_tot * 100
g['저녁비중']  = g['저녁매출'] / safe_tot * 100
g['야간비중']  = g['야간매출'] / safe_tot * 100
g['주말비중']  = g['주말매출'] / safe_tot * 100
g['여성비중']  = g['여성매출'] / safe_tot * 100
g['객단가']    = g['총매출']   / g['총건수'].replace(0, np.nan)
g['log_객단가'] = np.log(g['객단가'].replace(0, np.nan))   # 로그 변환
print(f"[A] 비율 피처 생성 완료 — 상권 수: {len(g)}")

# ── (B) 유동인구 성별·연령 비율 ────────────────────────────
# 기준: 총 유동인구 대비 성별·연령 비중으로 상권 고객 구성 파악
# 적용: jeongyeon_eda, dada, deep
if '남성_유동인구_수' in df_pop.columns:
    df_pop = df_pop.copy()
    safe_pop = df_pop['총_유동인구_수'].replace(0, np.nan)
    df_pop['남성_비율'] = df_pop['남성_유동인구_수'] / safe_pop * 100
    df_pop['여성_비율'] = df_pop['여성_유동인구_수'] / safe_pop * 100
    print("[B] 유동인구 성별 비율 생성 완료")

# ── (C) merge — 데이터 결합 ────────────────────────────────
# 기준: 상권코드(상권_코드)와 분기코드(기준_년분기_코드)를 복합키로 데이터 결합
# 주의: 좌우 dtype이 다르면 조인 누락 → merge 전 dtype 일치 확인 필수
# 적용: deep.ipynb (매출+영역+직장+상주+유동 통합)
매출_영역 = df_sales.merge(
    df_area[['상권_코드', '자치구_코드_명', '상권_구분_코드_명', '엑스좌표_값', '와이좌표_값']],
    on='상권_코드',
    how='left'
)
print(f"[C] 추정매출+영역 merge: {len(df_sales):,}행 → {len(매출_영역):,}행")

# ── (D) rename — 컬럼명 변경 ───────────────────────────────
# 기준: 다른 데이터셋과 merge할 때 컬럼명 충돌 방지
# 적용: jeongyeon_deep_data2.ipynb
# df_com = df_com.rename(columns={'자치구_코드_명': '자치구'})
print("[D] rename: 컬럼명 충돌 방지용 (jeongyeon_deep_data2 적용)")

# ── (E) 구간화(Binning) ────────────────────────────────────
# pd.cut  : 사업적 해석 기준이 있는 경우 (매출 규모)
# pd.qcut : 분포 기반 분위수 (유동인구 비율 5분위)
g['매출구간'] = pd.cut(
    g['총매출'] / 1e8,
    bins=[0, 10, 50, 150, 500, float('inf')],
    labels=['10억 미만', '10~50억', '50~150억', '150~500억', '500억 이상'],
    right=False
)
print(f"[E] 매출 구간화(pd.cut) 완료:\n{g['매출구간'].value_counts().sort_index().to_string()}")

# ── (F) 표준화 — KMeans 전처리 ─────────────────────────────
# 기준: KMeans는 유클리드 거리 기반 → 피처 스케일 차이에 민감
# 주의: NaN 포함 시 오류 → fillna(0) 후 스케일링
FEAT_COLS = ['점심비중', '저녁비중', '야간비중', '주말비중', '여성비중', 'log_객단가']
X = StandardScaler().fit_transform(g[FEAT_COLS].fillna(0))
print(f"\n[F] StandardScaler 표준화 완료 — shape: {X.shape}")

# ── (G) 좌표 변환: TM → WGS84 ─────────────────────────────
# 기준: 서울시 데이터 좌표가 TM(EPSG:5174) → Folium 지도용 WGS84로 변환
try:
    from pyproj import Transformer
    tr = Transformer.from_crs('EPSG:5174', 'EPSG:4326', always_xy=True)
    lons, lats = tr.transform(df_area['엑스좌표_값'].values, df_area['와이좌표_값'].values)
    df_area = df_area.copy()
    df_area['lat'], df_area['lon'] = lats, lons
    print(f"[G] 좌표 변환(TM→WGS84) 완료 — 위도: {lats.min():.3f}~{lats.max():.3f}")
except ImportError:
    print("[G] pyproj 미설치 (pip install pyproj)")

# ── (H) 코로나 구간 매핑 ───────────────────────────────────
# 기준: 분기코드를 코로나 단계별 레이블로 구간화 → 시기별 비교 분석
# 적용: jeongyeon_eda.ipynb, dada.ipynb
def covid_label(code):
    if 20191 <= code <= 20194: return '①코로나 전'
    elif 20201 <= code <= 20214: return '②코로나 초기'
    elif 20211 <= code <= 20224: return '③단계적 일상회복'
    elif 20231 <= code <= 20244: return '④엔데믹 이후'
    else: return '⑤2025'

if '기준_년분기_코드' in df_sales.columns:
    df_sales['코로나구간'] = df_sales['기준_년분기_코드'].apply(covid_label)
    print(f"[H] 코로나 구간 매핑 완료: {df_sales['코로나구간'].value_counts().to_dict()}")

print("\n✅ 데이터 변환 완료")


[A] 비율 피처 생성 완료 — 상권 수: 1603
[B] 유동인구 성별 비율 생성 완료
[C] 추정매출+영역 merge: 519,931행 → 519,931행
[D] rename: 컬럼명 충돌 방지용 (jeongyeon_deep_data2 적용)
[E] 매출 구간화(pd.cut) 완료:
매출구간
10억 미만       50
10~50억      109
50~150억     140
150~500억    368
500억 이상     936

[F] StandardScaler 표준화 완료 — shape: (1603, 6)
[G] 좌표 변환(TM→WGS84) 완료 — 위도: 37.437~37.693
[H] 코로나 구간 매핑 완료: {'④엔데믹 이후': 175425, '②코로나 초기': 169940, '③단계적 일상회복': 88834, '⑤2025': 85732}

✅ 데이터 변환 완료


## 6. RFM 지표 생성 및 상권 분류

In [7]:
# ══════════════════════════════════════════════════════════
# [전처리 6] RFM 기초 지표 계산
# ══════════════════════════════════════════════════════════
#
# R (Recency)   : 매출 증감률 — 얼마나 최근에 성장했는가
#                 = (2025 매출 - 2024 매출) / 2024 매출 × 100
# F (Frequency) : 결제건수    — 얼마나 자주 거래가 발생하는가
#                 = 2025년 총 결제건수
# M (Monetary)  : 객단가      — 1건당 평균 매출이 얼마인가
#                 = 2025 매출 / 2025 결제건수
#
# 기준: 2024 vs 2025 비교로 최근성 측정
#       연간 합산 사용 (분기별 계절성 노이즈 제거)
# ══════════════════════════════════════════════════════════

# ── 연도 컬럼 추가
df_sales['연도'] = df_sales['기준_년분기_코드'] // 10

# ── 연도별 상권 집계
연도별 = (
    df_sales.groupby(['상권_코드', '연도'])
    .agg(매출=('당월_매출_금액', 'sum'),
         결제건수=('당월_매출_건수', 'sum'))
    .reset_index()
)

# ── 2024 / 2025 분리 후 피벗
매출_2024 = 연도별[연도별['연도'] == 2024].set_index('상권_코드')[['매출', '결제건수']]
매출_2025 = 연도별[연도별['연도'] == 2025].set_index('상권_코드')[['매출', '결제건수']]
매출_2024.columns = ['매출_2024', '결제건수_2024']
매출_2025.columns = ['매출_2025', '결제건수_2025']

rfm_base = 매출_2024.join(매출_2025, how='inner').reset_index()

# ── 평균 유동인구 (2025년 기준)
if '기준_년분기_코드' in df_pop.columns:
    팝_2025 = df_pop[df_pop['기준_년분기_코드'] // 10 == 2025]
    유동평균 = 팝_2025.groupby('상권_코드')['총_유동인구_수'].mean().rename('평균_유동인구')
    rfm_base = rfm_base.merge(유동평균, on='상권_코드', how='left')

# ── 영역 정보 (상권명, 자치구, 좌표) 결합
rfm_base = rfm_base.merge(
    df_area[['상권_코드', '상권_코드_명', '자치구_코드_명', '상권_구분_코드_명', '엑스좌표_값', '와이좌표_값']],
    on='상권_코드', how='left'
)

# ── R·F·M 지표 계산
# R: 매출 증감률 (2024→2025, %)
rfm_base['R_매출증감률']  = (rfm_base['매출_2025'] - rfm_base['매출_2024']) / rfm_base['매출_2024'].replace(0, float('nan')) * 100
rfm_base['결제건수_증감률'] = (rfm_base['결제건수_2025'] - rfm_base['결제건수_2024']) / rfm_base['결제건수_2024'].replace(0, float('nan')) * 100

# F: 2025년 총 결제건수
rfm_base['F_결제건수'] = rfm_base['결제건수_2025']

# M: 객단가 (매출 / 결제건수)
rfm_base['M_객단가'] = rfm_base['매출_2025'] / rfm_base['결제건수_2025'].replace(0, float('nan'))

# 결제전환율: 결제건수 / 유동인구 (상권 구매 효율)
if '평균_유동인구' in rfm_base.columns:
    rfm_base['결제전환율'] = rfm_base['F_결제건수'] / rfm_base['평균_유동인구'].replace(0, float('nan'))

# 진짜성장 플래그: 매출↑ AND 결제건수↑ 동시 증가 → 인플레이션 제거 성장
rfm_base['진짜성장_플래그'] = (rfm_base['R_매출증감률'] > 0) & (rfm_base['결제건수_증감률'] > 0)

print(f"RFM 기초 테이블 생성 완료: {rfm_base.shape}")
print(rfm_base[['상권_코드_명', 'R_매출증감률', 'F_결제건수', 'M_객단가']].head(5).round(2).to_string())


RFM 기초 테이블 생성 완료: (1575, 17)
                  상권_코드_명  R_매출증감률    F_결제건수     M_객단가
0                이태원 관광특구    -3.14  12497758  28572.36
1  명동 남대문 북창동 다동 무교동 관광특구     4.14  60052806  32439.17
2            동대문패션타운 관광특구    13.36  16200152  33802.86
3              종로?청계 관광특구    -2.32  27910035  35067.24
4                 잠실 관광특구     1.95  54650346  26898.27


In [8]:
# ══════════════════════════════════════════════════════════
# [전처리 7] RFM 점수 부여 및 상권 분류
# ══════════════════════════════════════════════════════════

# ── 결측치 제거 (R·F·M 3개 지표 중 하나라도 없으면 제외)
# 기준: 점수 계산 자체가 불가능 → 분석 대상에서 제외
rfm_clean = rfm_base[
    rfm_base[['R_매출증감률', 'F_결제건수', 'M_객단가']].notna().all(axis=1)
].copy()
print(f"분석 대상: {len(rfm_clean)}개 / 전체 {len(rfm_base)}개 (제외: {len(rfm_base)-len(rfm_clean)}개)")

# ── 5분위수(quintile) 점수 부여 (1~5점)
# 기준: 절대값 기준이 아닌 상대 순위로 점수 → 상권 간 공정 비교 가능
#       duplicates='drop': 값이 같은 경계값이 있어도 오류 없이 처리
rfm_clean['R_점수'] = pd.qcut(rfm_clean['R_매출증감률'], q=5, labels=[1,2,3,4,5], duplicates='drop').astype(int)
rfm_clean['F_점수'] = pd.qcut(rfm_clean['F_결제건수'],    q=5, labels=[1,2,3,4,5], duplicates='drop').astype(int)
rfm_clean['M_점수'] = pd.qcut(rfm_clean['M_객단가'],      q=5, labels=[1,2,3,4,5], duplicates='drop').astype(int)
rfm_clean['RFM_합계'] = rfm_clean['R_점수'] + rfm_clean['F_점수'] + rfm_clean['M_점수']

print(f"\nRFM 점수 분포:")
print(rfm_clean[['R_점수','F_점수','M_점수','RFM_합계']].describe().round(2).to_string())

# ── 성장유형 분류 (매출증감률 × 결제건수증감률 조합)
# 기준: 매출↑이 인플레이션인지 진짜 성장인지 구분
#   진짜성장         : 매출↑ AND 결제건수↑  (고객 수 + 지출 모두 증가)
#   가짜성장(인플레이션): 매출↑ AND 결제건수↓  (고객은 줄었는데 1인당 지출↑)
#   저가성장         : 매출↓ AND 결제건수↑  (고객은 많지만 객단가 하락)
#   침체             : 매출↓ AND 결제건수↓  (모두 감소)
rfm_clean['성장유형'] = '침체'
rfm_clean.loc[(rfm_clean['R_매출증감률'] > 0) & (rfm_clean['결제건수_증감률'] > 0),  '성장유형'] = '진짜성장'
rfm_clean.loc[(rfm_clean['R_매출증감률'] > 0) & (rfm_clean['결제건수_증감률'] <= 0), '성장유형'] = '가짜성장(인플레이션)'
rfm_clean.loc[(rfm_clean['R_매출증감률'] <= 0) & (rfm_clean['결제건수_증감률'] > 0), '성장유형'] = '저가성장'
print(f"\n성장유형 분포:\n{rfm_clean['성장유형'].value_counts().to_string()}")

# ── 상권 분류 (RFM 점수 조합 기준)
# 기준 (우선순위 순 적용):
#   스타_상권    : R≥4 AND F≥4 AND M≥3  → 매출 성장 + 고객 많음 + 객단가 높음
#   위험_상권    : R≤2 AND F≤2 AND M≤2  → 3개 지표 모두 하위
#   고평가_버블  : F≥4 AND M≤2           → 고객 많지만 객단가 낮음 (트래픽 함정)
#   저평가_가치주: F≤2 AND M≥4           → 고객 적지만 객단가 높음 (알짜배기 상권)
#   일반_상권    : 나머지
rfm_clean['상권분류'] = '일반_상권'
rfm_clean.loc[(rfm_clean['F_점수'] <= 2) & (rfm_clean['M_점수'] >= 4), '상권분류'] = '저평가_가치주'
rfm_clean.loc[(rfm_clean['F_점수'] >= 4) & (rfm_clean['M_점수'] <= 2), '상권분류'] = '고평가_버블'
rfm_clean.loc[(rfm_clean['R_점수'] <= 2) & (rfm_clean['F_점수'] <= 2) & (rfm_clean['M_점수'] <= 2), '상권분류'] = '위험_상권'
rfm_clean.loc[(rfm_clean['R_점수'] >= 4) & (rfm_clean['F_점수'] >= 4) & (rfm_clean['M_점수'] >= 3), '상권분류'] = '스타_상권'

print(f"\n상권 분류 결과:\n{rfm_clean['상권분류'].value_counts().to_string()}")

# ── 세그먼트별 프로파일
print("\n=== RFM 세그먼트 프로파일 ===")
print(rfm_clean.groupby('상권분류')[['R_매출증감률','F_결제건수','M_객단가','RFM_합계']].mean().round(2).to_string())


분석 대상: 1575개 / 전체 1575개 (제외: 0개)

RFM 점수 분포:
          R_점수     F_점수     M_점수   RFM_합계
count  1575.00  1575.00  1575.00  1575.00
mean      3.00     3.00     3.00     9.00
std       1.41     1.41     1.41     2.34
min       1.00     1.00     1.00     3.00
25%       2.00     2.00     2.00     7.00
50%       3.00     3.00     3.00     9.00
75%       4.00     4.00     4.00    11.00
max       5.00     5.00     5.00    15.00

성장유형 분포:
성장유형
침체             821
진짜성장           373
가짜성장(인플레이션)    274
저가성장           107

상권 분류 결과:
상권분류
일반_상권      671
저평가_가치주    370
고평가_버블     299
스타_상권      159
위험_상권       76

=== RFM 세그먼트 프로파일 ===
         R_매출증감률      F_결제건수     M_객단가  RFM_합계
상권분류                                          
고평가_버블     -1.09  3347994.03  17632.98    9.10
스타_상권      20.46  5466484.06  35465.50   12.67
위험_상권     -24.47   159728.30  15013.96    4.37
일반_상권       4.34  1693504.35  28723.33    8.61
저평가_가치주    96.48    92045.30  96823.51    9.00


In [9]:
# ══════════════════════════════════════════════════════════
# [전처리 8] RFM 결과 저장
# ══════════════════════════════════════════════════════════
from pathlib import Path

OUTPUT_DIR = project_root / 'data' / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

# 기초 테이블 저장
rfm_base.to_csv(OUTPUT_DIR / '상권별_RFM기초.csv', index=False, encoding='utf-8-sig')

# 점수 + 분류 포함 최종 결과 저장
저장_컬럼 = ['상권_코드', '상권_코드_명', '자치구_코드_명', '상권_구분_코드_명',
             'R_매출증감률', '결제건수_증감률', 'F_결제건수', 'M_객단가',
             'R_점수', 'F_점수', 'M_점수', 'RFM_합계',
             '성장유형', '상권분류', '진짜성장_플래그',
             '결제전환율', '엑스좌표_값', '와이좌표_값']
존재_컬럼 = [c for c in 저장_컬럼 if c in rfm_clean.columns]
rfm_clean[존재_컬럼].to_csv(OUTPUT_DIR / 'RFM_최종분석결과.csv', index=False, encoding='utf-8-sig')

print(f"✅ RFM 결과 저장 완료")
print(f"  상권별_RFM기초.csv     : {len(rfm_base):,}개 상권")
print(f"  RFM_최종분석결과.csv   : {len(rfm_clean):,}개 상권 (결측 제거 후)")
print(f"  저장 위치: {OUTPUT_DIR}")


✅ RFM 결과 저장 완료
  상권별_RFM기초.csv     : 1,575개 상권
  RFM_최종분석결과.csv   : 1,575개 상권 (결측 제거 후)
  저장 위치: ..\data\output


## 전처리 결과 요약

In [10]:
# ══════════════════════════════════════════════════════════
# 전처리 결과 요약 출력
# ══════════════════════════════════════════════════════════

summary = {
    '결측치': [
        'fillna(0)          — 업종 없는 상권 매출 (매출 부재 = 0원)',
        'fillna("서울 외")  — 지하철역 자치구 미매핑 (jeongyeon)',
        'dropna(subset)     — 소득 데이터 결측 행 (jeongyeon_deep3)',
        'replace(0, np.nan) — 비율 분모=0 방지',
        'notna().all(axis=1)— RFM 지표 NaN 행 제거',
    ],
    '이상치': [
        'describe() 기술통계 확인 — 전체 데이터셋',
        'IQR 방법             — RFM 지표 극단치 탐지',
        '이상치 제거 안 함    — 상권 매출(100배 차이가 정상)',
        'np.log / np.log1p   — 로그 변환으로 분포 보정',
        '분기 필터링 <20250  — 2025 미완성 데이터 제외',
        '최소 8분기 필터링   — 취약상권 신뢰도 기준',
        '80~99% 축소 필터링 — 취약상권 정의 기준',
    ],
    '중복': [
        'duplicated().sum()  — 전체 데이터셋 확인',
        'drop_duplicates(subset=복합키) — 분기+상권+업종',
    ],
    '타입변환': [
        '기준_년분기_코드: int → //10, %10 → 연도·분기',
        '사용월(YYYYMM): str[:4], str[4:6] → 연도·월 → 분기',
        'pd.to_numeric(errors="coerce") — 비숫자 "-" → NaN',
        'pd.PeriodIndex(freq="Q")       — 시계열 연산용',
        'str.replace("?", "·")         — 인코딩 깨진 특수문자',
    ],
    '데이터변환': [
        '비율 피처 — 시간대·성별·연령 매출 비중 (%)',
        'merge(on=[분기코드, 상권코드]) — 데이터 결합',
        'groupby().agg() — 상권별·연도별 집계',
        'pd.cut / pd.qcut — 매출구간화 / 유동인구 5분위',
        'StandardScaler  — KMeans 전처리 표준화',
        '좌표변환 TM→WGS84 — Folium 지도 표시',
        '코로나 구간 레이블 — 시기별 비교 분석',
    ],
}

print("=" * 60)
print("  서울DF 프로젝트 — 전처리 결과 요약")
print("=" * 60)
for 항목, 내용들 in summary.items():
    print(f"\n【{항목}】")
    for 내용 in 내용들:
        print(f"  ✓ {내용}")
print("\n" + "=" * 60)


  서울DF 프로젝트 — 전처리 결과 요약

【결측치】
  ✓ fillna(0)          — 업종 없는 상권 매출 (매출 부재 = 0원)
  ✓ fillna("서울 외")  — 지하철역 자치구 미매핑 (jeongyeon)
  ✓ dropna(subset)     — 소득 데이터 결측 행 (jeongyeon_deep3)
  ✓ replace(0, np.nan) — 비율 분모=0 방지
  ✓ notna().all(axis=1)— RFM 지표 NaN 행 제거

【이상치】
  ✓ describe() 기술통계 확인 — 전체 데이터셋
  ✓ IQR 방법             — RFM 지표 극단치 탐지
  ✓ 이상치 제거 안 함    — 상권 매출(100배 차이가 정상)
  ✓ np.log / np.log1p   — 로그 변환으로 분포 보정
  ✓ 분기 필터링 <20250  — 2025 미완성 데이터 제외
  ✓ 최소 8분기 필터링   — 취약상권 신뢰도 기준
  ✓ 80~99% 축소 필터링 — 취약상권 정의 기준

【중복】
  ✓ duplicated().sum()  — 전체 데이터셋 확인
  ✓ drop_duplicates(subset=복합키) — 분기+상권+업종

【타입변환】
  ✓ 기준_년분기_코드: int → //10, %10 → 연도·분기
  ✓ 사용월(YYYYMM): str[:4], str[4:6] → 연도·월 → 분기
  ✓ pd.to_numeric(errors="coerce") — 비숫자 "-" → NaN
  ✓ pd.PeriodIndex(freq="Q")       — 시계열 연산용
  ✓ str.replace("?", "·")         — 인코딩 깨진 특수문자

【데이터변환】
  ✓ 비율 피처 — 시간대·성별·연령 매출 비중 (%)
  ✓ merge(on=[분기코드, 상권코드]) — 데이터 결합
  ✓ groupby().agg() — 상권별·연도별 집계
  ✓ pd.cut / pd.qcut — 매출구간화 / 유동인구 5분위
  ✓ Stand